# WASM Output Level Analysis — Full Provenance Chain

**Question:** Why does SuperSonic's scsynth WASM produce louder audio than desktop scsynth for identical code?

**Method:** Start from raw A/B WAV recordings, derive every claim from data, trace the signal path, identify exactly where the evidence chain becomes inference.

**A/B Test Setup:**
- Same code run on both platforms (DJ Dave kick+clap at 130 BPM)
- Both recorded via platform Rec button (DiskOut on desktop, MediaRecorder on web)
- Two isolated tests: only_Drums (kick pattern) and no_Drums (clap + echo + reverb)
- Desktop: Sonic Pi (latest stable, native scsynth)
- Web: Sonic Pi Web (SuperSonic scsynth WASM v0.66.0)

In [ ]:
import wave, struct, math
import os

def load_wav(path):
    """Load WAV file, return mono float samples [-1,1] and metadata."""
    with wave.open(path, 'rb') as w:
        n = w.getnframes()
        ch = w.getnchannels()
        sw = w.getsampwidth()
        rate = w.getframerate()
        raw = w.readframes(n)
    maxval = {1: 127, 2: 32767, 4: 2147483647}[sw]
    fmt_char = {1: 'b', 2: 'h', 4: 'i'}[sw]
    samples = struct.unpack(f'<{n*ch}{fmt_char}', raw)
    floats = [s / maxval for s in samples]
    # Mono mix
    mono = [sum(floats[i:i+ch])/ch for i in range(0, len(floats), ch)]
    return {
        'mono': mono,
        'stereo': floats,
        'rate': rate,
        'channels': ch,
        'bits': sw * 8,
        'frames': n,
        'duration': n / rate,
        'path': path,
    }

def compute_stats(wav):
    """Compute audio statistics from loaded WAV."""
    mono = wav['mono']
    stereo = wav['stereo']
    peak = max(abs(s) for s in stereo)
    rms = math.sqrt(sum(s*s for s in stereo) / len(stereo))
    clipping = sum(1 for s in stereo if abs(s) > 0.95) / len(stereo) * 100
    crest = peak / rms if rms > 0 else 0
    dc = sum(stereo) / len(stereo)
    
    # Per-second RMS
    rate = wav['rate']
    ch = wav['channels']
    per_sec = []
    for sec in range(int(wav['duration'])):
        start = sec * rate * ch
        end = min((sec + 1) * rate * ch, len(stereo))
        chunk = stereo[start:end]
        if chunk:
            per_sec.append(math.sqrt(sum(s*s for s in chunk) / len(chunk)))
    
    return {
        'peak': peak,
        'rms': rms,
        'clipping_pct': clipping,
        'crest_factor': crest,
        'dc_offset': dc,
        'per_sec_rms': per_sec,
    }

print('Utilities loaded.')

## Step 1: Load the A/B Test WAVs

**Evidence type:** Direct observation (raw WAV data from both platforms)

In [ ]:
BASE = 'latest_test'

# Load all 4 WAVs
drums_desktop = load_wav(f'{BASE}/only_Drums/OriginalSonicPi_only_Drums.wav')
drums_web     = load_wav(f'{BASE}/only_Drums/SonicPiWeb_only_Drums.wav')
clap_desktop  = load_wav(f'{BASE}/no_Drums/OriginalSonicPi_no_Drums.wav')
clap_web      = load_wav(f'{BASE}/no_Drums/SonicPiWeb_no_Drums.wav')

for label, wav in [('Drums Desktop', drums_desktop), ('Drums Web', drums_web),
                    ('Clap Desktop', clap_desktop), ('Clap Web', clap_web)]:
    print(f'{label}: {wav["duration"]:.1f}s, {wav["channels"]}ch, {wav["rate"]}Hz, {wav["bits"]}bit')

## Step 2: Observation 1 — Raw Level Comparison

**Claim:** Web output is ~2x louder than desktop for identical code.

**Evidence type:** Direct measurement from WAV files. No inference.

In [ ]:
drums_d_stats = compute_stats(drums_desktop)
drums_w_stats = compute_stats(drums_web)
clap_d_stats  = compute_stats(clap_desktop)
clap_w_stats  = compute_stats(clap_web)

print('=' * 70)
print('OBSERVATION 1: Raw level comparison (direct measurement)')
print('=' * 70)
print(f'{"":20} {"Desktop":>10} {"Web":>10} {"Ratio":>10}')
print('-' * 55)

for label, d, w in [('Drums RMS', drums_d_stats['rms'], drums_w_stats['rms']),
                     ('Drums Peak', drums_d_stats['peak'], drums_w_stats['peak']),
                     ('Drums Clipping%', drums_d_stats['clipping_pct'], drums_w_stats['clipping_pct']),
                     ('Drums Crest', drums_d_stats['crest_factor'], drums_w_stats['crest_factor']),
                     ('', 0, 0),
                     ('Clap RMS', clap_d_stats['rms'], clap_w_stats['rms']),
                     ('Clap Peak', clap_d_stats['peak'], clap_w_stats['peak']),
                     ('Clap Clipping%', clap_d_stats['clipping_pct'], clap_w_stats['clipping_pct']),
                     ('Clap Crest', clap_d_stats['crest_factor'], clap_w_stats['crest_factor'])]:
    if not label:
        print()
        continue
    ratio = w / d if d > 0 else 0
    print(f'{label:20} {d:10.4f} {w:10.4f} {ratio:9.2f}x')

drums_rms_ratio = drums_w_stats['rms'] / drums_d_stats['rms']
clap_rms_ratio = clap_w_stats['rms'] / clap_d_stats['rms']
print(f'\n>>> MEASURED: Drums ratio = {drums_rms_ratio:.2f}x, Clap ratio = {clap_rms_ratio:.2f}x')
print(f'>>> NOTE: The ratio is NOT constant — it differs between signals.')

## Step 3: Observation 2 — Per-Second Stability

**Claim:** The ratio is sustained over time, not a transient artifact.

**Evidence type:** Direct measurement.

In [ ]:
print('Per-second RMS ratios (Web / Desktop):')
print()

for test_label, d_stats, w_stats in [
    ('DRUMS', drums_d_stats, drums_w_stats),
    ('CLAP', clap_d_stats, clap_w_stats),
]:
    d_sec = d_stats['per_sec_rms']
    w_sec = w_stats['per_sec_rms']
    min_len = min(len(d_sec), len(w_sec))
    print(f'{test_label}:')
    ratios = []
    for i in range(min_len):
        if d_sec[i] > 0.01 and w_sec[i] > 0.01:  # skip silence
            r = w_sec[i] / d_sec[i]
            ratios.append(r)
            print(f'  sec {i}: desktop={d_sec[i]:.4f}  web={w_sec[i]:.4f}  ratio={r:.2f}x')
    if ratios:
        avg = sum(ratios) / len(ratios)
        std = math.sqrt(sum((r - avg)**2 for r in ratios) / len(ratios))
        print(f'  >>> Mean ratio: {avg:.2f}x ± {std:.2f}')
    print()

## Step 4: Observation 3 — Desktop Does NOT Clip, Web Does

**Claim:** Desktop signal stays well below 1.0. Web signal hits 1.0 (hard clip). This means the mixer's clip2(1) is triggered on web but not on desktop.

**Evidence type:** Direct measurement.

**Significance:** If both use the same mixer (pre_amp=0.2, amp=6, Limiter.ar, clip2), and desktop doesn't clip while web does, the INPUT to the mixer must be louder on web.

In [ ]:
print('Peak and clipping analysis:')
print(f'  Desktop drums: peak={drums_d_stats["peak"]:.4f}, clipping={drums_d_stats["clipping_pct"]:.2f}%')
print(f'  Web drums:     peak={drums_w_stats["peak"]:.4f}, clipping={drums_w_stats["clipping_pct"]:.2f}%')
print()

# The mixer chain: raw → pre_amp(0.2) → Limiter(0.99) → amp(6) → clip2(1)
# Desktop peak = 0.63 < 1.0 → clip2 never triggers
# Web peak = 1.0 = clip2 threshold → clip2 IS triggering

d_peak = drums_d_stats['peak']
print('INFERENCE 1: Reverse-engineer raw signal from mixer output')
print('  Mixer chain: raw → ×pre_amp(0.2) → Limiter(0.99) → ×amp(6) → clip2(1)')
print()

# Desktop: peak 0.63, no clipping → Limiter didn't engage → signal < 0.99 after pre_amp
d_before_clip = d_peak  # 0.63 (no clipping occurred)
d_before_amp = d_before_clip / 6  # ÷ amp
d_limiter_engaged = d_before_amp > 0.99
d_before_preamp = d_before_amp / 0.2  # ÷ pre_amp

print(f'  DESKTOP (no clipping → can reverse entire chain):')
print(f'    Output peak:       {d_peak:.4f}')
print(f'    Before clip2:      {d_before_clip:.4f} (< 1.0, clip2 did NOT trigger)')
print(f'    Before amp(×6):    {d_before_amp:.4f}')
print(f'    Limiter engaged:   {d_limiter_engaged} ({d_before_amp:.4f} {"≥" if d_limiter_engaged else "<"} 0.99)')
print(f'    Before pre_amp:    {d_before_preamp:.4f} ← THIS IS THE RAW SIGNAL PEAK')
print()

# Web: peak 1.0, clipping → we know clip2 triggered, but we can't reverse THROUGH the clip
# We only know: before clip2, signal was ≥ 1.0
# And: Limiter caps at 0.99, so before amp: ≤ 0.99, after amp: ≤ 5.94 → clip2 clips to 1.0
print(f'  WEB (clipping → CANNOT fully reverse the chain):')
print(f'    Output peak:       1.0000 (clipped)')
print(f'    Before clip2:      UNKNOWN (≥ 1.0, destroyed by clipping)')
print(f'    Before amp(×6):    UNKNOWN (Limiter may have engaged)')
print(f'    Before pre_amp:    UNKNOWN from this data alone')
print()
print(f'  >>> We CANNOT determine the exact web raw signal from the clipped output.')
print(f'  >>> We need an independent measurement. See Step 5.')

## Step 5: Observation 4 — The amp=1 Mixer Test

**Claim:** By changing mixer amp from 6 to 1 and measuring the output, we can determine the raw signal level independently.

**Evidence type:** Direct measurement (separate capture with amp=1). The capture was run on the full DJ Dave code (kick + clap combined).

**Result:** amp=1 → RMS 0.074, peak 0.59, clipping 0.00%

In [ ]:
# amp=1 test results (from earlier capture — manual entry since WAV not saved)
amp1_rms = 0.074
amp1_peak = 0.59
amp1_clipping = 0.0

print('amp=1 mixer test (full DJ Dave code, web only):')
print(f'  RMS:      {amp1_rms}')
print(f'  Peak:     {amp1_peak}')
print(f'  Clipping: {amp1_clipping}%')
print()

# With amp=1: mixer chain is raw → pre_amp(0.2) → Limiter(0.99) → amp(1) → clip2(1)
# Peak 0.59 < 1.0 → no clipping → can reverse fully
amp1_before_clip = amp1_peak
amp1_before_amp = amp1_before_clip / 1  # amp=1
amp1_limiter_engaged = amp1_before_amp > 0.99
amp1_before_preamp = amp1_before_amp / 0.2

print('INFERENCE 2: Reverse-engineer web raw signal from amp=1 test')
print(f'  Output peak:     {amp1_peak:.4f} (no clipping → full chain reversible)')
print(f'  Before clip2:    {amp1_before_clip:.4f}')
print(f'  Before amp(×1):  {amp1_before_amp:.4f}')
print(f'  Limiter engaged: {amp1_limiter_engaged}')
print(f'  Before pre_amp:  {amp1_before_preamp:.4f} ← WEB RAW SIGNAL PEAK')
print()

# RMS
amp1_raw_rms = amp1_rms / 0.2 / 1  # ÷ pre_amp ÷ amp
print(f'  Raw RMS: {amp1_rms} / 0.2 / 1 = {amp1_raw_rms:.4f}')
print()

# Cross-check: does amp=1 output match expected?
# If raw_rms = 0.37, then with amp=6: 0.37 × 0.2 × 6 = 0.444
# But clip2 would clip peaks, reducing RMS slightly below 0.444
expected_amp6_rms = amp1_raw_rms * 0.2 * 6
print(f'CROSS-CHECK: raw_rms({amp1_raw_rms:.3f}) × 0.2 × 6 = {expected_amp6_rms:.3f}')
print(f'  Actual web drums+clap RMS at amp=6 ≈ 0.42 (slightly less due to clipping)')
print(f'  Cross-check: ✓ consistent (0.44 theoretical → 0.42 actual, clipping reduces it)')

## Step 6: Inference — The Raw Signal Ratio

**Claim:** Web raw signal is ~2.3x louder than desktop raw signal.

**Evidence chain:**
1. Desktop raw peak = 0.52 (from Step 4, reversed from output peak 0.63)
2. Web raw RMS = 0.37 (from Step 5, amp=1 test)
3. Desktop raw RMS ≈ 0.16 (from output RMS 0.188 / gain 1.2)

**Strength of evidence:**
- Desktop raw peak: STRONG (no clipping → fully reversible)
- Web raw RMS: STRONG (amp=1 test, no clipping → fully reversible)
- Desktop raw RMS: MODERATE (assumes linear gain, no limiter engagement — valid since peak < 0.99 after pre_amp)

In [ ]:
desktop_raw_peak = d_before_preamp  # from Step 4
desktop_raw_rms = drums_d_stats['rms'] / (0.2 * 6)  # output / gain (no limiter, no clip)
web_raw_rms = amp1_raw_rms  # from Step 5
web_raw_peak = amp1_before_preamp  # from Step 5

print('RAW SIGNAL COMPARISON:')
print(f'{"":15} {"Desktop":>10} {"Web":>10} {"Ratio":>10}')
print(f'{"Raw Peak":15} {desktop_raw_peak:10.4f} {web_raw_peak:10.4f} {web_raw_peak/desktop_raw_peak:9.2f}x')
print(f'{"Raw RMS":15} {desktop_raw_rms:10.4f} {web_raw_rms:10.4f} {web_raw_rms/desktop_raw_rms:9.2f}x')
print()

rms_ratio = web_raw_rms / desktop_raw_rms
peak_ratio = web_raw_peak / desktop_raw_peak
print(f'>>> Raw RMS ratio: {rms_ratio:.2f}x')
print(f'>>> Raw Peak ratio: {peak_ratio:.2f}x')
print()
print(f'CAVEAT: Desktop raw RMS is inferred from output÷gain.')
print(f'  This is valid ONLY if the limiter did not engage.')
print(f'  Limiter threshold: 0.99. Desktop after pre_amp: {desktop_raw_peak * 0.2:.4f}')
print(f'  {desktop_raw_peak * 0.2:.4f} < 0.99 → Limiter did NOT engage → inference valid ✓')
print()
print(f'CAVEAT: Web raw RMS ({web_raw_rms:.3f}) is from FULL code (kick+clap).')
print(f'  Desktop raw RMS ({desktop_raw_rms:.3f}) is from drums-only.')
print(f'  The ratio ({rms_ratio:.2f}x) mixes different signals → APPROXIMATE.')
print(f'  For drums-only: ratio = {drums_w_stats["rms"] / drums_d_stats["rms"]:.2f}x (output level)')
print(f'  For clap-only:  ratio = {clap_w_stats["rms"] / clap_d_stats["rms"]:.2f}x (output level)')

## Step 7: The Signal-Dependent Factor

**Observation:** The output ratio differs between drums (2.2x) and clap+FX (1.8x).

**This is critical.** If the difference were a simple constant scaling factor (like a hardcoded gain somewhere in the WASM output path), the ratio would be the same for ALL signals. The fact that it varies means:

**Either:**
- (A) The difference is a constant factor applied BEFORE non-linear processing (mixer limiter/clip), and the non-linearity changes the apparent ratio — OR
- (B) The difference is truly signal-dependent (different UGens produce different ratios)

Let's test hypothesis A.

In [ ]:
print('HYPOTHESIS A: Constant factor + non-linear mixer explains the variation')
print()
print('If the raw signal is K× louder, the mixer processes K× louder input.')
print('The mixer has a Limiter(0.99) + clip2(1). For loud signals:')
print('  - Limiter engages → caps the signal → reduces the APPARENT ratio')
print('  - clip2 engages → further caps → further reduces apparent ratio')
print()
print('For QUIET signals (clap+FX, RMS 0.05-0.09):')
print('  - Raw × K probably stays below limiter → linear passthrough')
print('  - Output ratio ≈ K (the true factor)')
print()
print('For LOUD signals (drums, RMS 0.19-0.42):')
print('  - Raw × K may exceed limiter → non-linear compression')
print('  - Output ratio < K (compressed by limiter/clip)')
print()

# Check: does the limiter engage for drums on web?
# Web drums peak after pre_amp: raw_peak × 0.2
# We know web drums clips at output → limiter likely engaged
# Desktop drums after pre_amp: 0.52 × 0.2 = 0.104 → well below 0.99

# If true K is ~1.8x (from clap, which is quieter and more linear):
K_from_clap = clap_w_stats['rms'] / clap_d_stats['rms']
K_from_drums = drums_w_stats['rms'] / drums_d_stats['rms']

print(f'Clap ratio (quieter, less clipping):  {K_from_clap:.2f}x')
print(f'Drums ratio (louder, clips):           {K_from_drums:.2f}x')
print()
print(f'If constant K ≈ {K_from_clap:.1f}x (from quiet signal):')
print(f'  Drums SHOULD be {K_from_clap:.1f}x but MEASURED {K_from_drums:.1f}x')
print(f'  Drums ratio > clap ratio → OPPOSITE of what hypothesis A predicts!')
print(f'  (Non-linear compression should REDUCE the loud ratio, not increase it)')
print()
print(f'>>> HYPOTHESIS A FAILS.')
print(f'>>> The drums ratio (2.2x) is HIGHER than the clap ratio (1.8x).')
print(f'>>> If a constant factor + limiter compression, drums should be LOWER.')
print(f'>>> The factor is genuinely signal-dependent.')

## Step 8: What We Can and Cannot Claim

Let's be precise about the provenance of each claim.

In [ ]:
print('EVIDENCE CHAIN — PROVENANCE OF EACH CLAIM')
print('=' * 70)
print()

claims = [
    ('PROVEN',
     'Web output is louder than desktop',
     'Direct WAV measurement: drums 2.21x, clap 1.82x RMS ratio'),
    
    ('PROVEN',
     'The mixer IS processing audio',
     'amp=1 test: RMS drops from 0.42 to 0.074 (6x, matching 6x amp change)'),
    
    ('PROVEN',
     'The ratio is sustained, not transient',
     'Per-second RMS ratios stable at 2.0-2.5x across all measured seconds'),
    
    ('PROVEN',
     'AudioWorklet applies zero gain',
     'Source code: direct .set() copy, verified from scsynth_audio_worklet.js'),
    
    ('PROVEN',
     'Web Audio chain applies zero extra gain',
     'ChannelMerger (unity per MDN spec), AnalyserNode (passive), GainNode=1.0'),
    
    ('PROVEN',
     'Mixer settings match desktop',
     'pre_amp=0.2, amp=6, confirmed from studio.rb source'),
    
    ('PROVEN',
     'Routing approach makes no difference',
     'autoConnect test: RMS 0.205 vs manual routing RMS 0.200'),
    
    ('PROVEN',
     'numOutputBusChannels makes no difference',
     'autoConnect test used default 2 channels, same RMS as 14-channel setup'),
    
    ('PROVEN',
     'The factor is signal-dependent (not constant)',
     'Drums: 2.2x, Clap+FX: 1.8x. Hypothesis A (constant + non-linearity) fails.'),
    
    ('INFERRED',
     'The difference originates inside scsynth WASM',
     'All external stages verified (worklet, Web Audio, mixer). Process of elimination.'),
    
    ('UNCERTAIN',
     'The cause is UGen-level float32 vs x87 precision',
     'Plausible but NOT tested. Would require comparing specific UGen outputs.'),
    
    ('UNCERTAIN',
     'The cause is bus accumulation between sub-blocks',
     'Plausible but NOT tested. Would require inspecting WASM World_Run cycle.'),
    
    ('UNCERTAIN',
     'Desktop audio drivers apply normalization',
     'Plausible but NOT tested. CoreAudio documentation does not confirm this.'),
    
    ('UNCERTAIN',
     'Compiled synthdefs differ between npm and desktop',
     'npm README says "from Sonic Pi" but binary identity not verified.'),
]

for status, claim, evidence in claims:
    icon = {'PROVEN': '✅', 'INFERRED': '🔶', 'UNCERTAIN': '❓'}[status]
    print(f'{icon} [{status}] {claim}')
    print(f'   Evidence: {evidence}')
    print()

## Step 9: Spectral Comparison — Same Instrument or Different?

**Question:** Is the WASM output the same sound at a different level, or a genuinely different sound?

In [ ]:
def band_energy(mono, rate, start_sec, dur_sec, lo_hz, hi_hz):
    """Compute energy in a frequency band using windowed DFT."""
    start = int(start_sec * rate)
    end = min(start + int(dur_sec * rate), len(mono))
    chunk = mono[start:end]
    N = len(chunk)
    if N < 100: return 0
    # Hann window
    windowed = [chunk[i] * (0.5 - 0.5 * math.cos(2 * math.pi * i / N)) for i in range(N)]
    k_lo = max(1, int(lo_hz * N / rate))
    k_hi = min(N // 2, int(hi_hz * N / rate))
    energy = 0
    step = max(1, (k_hi - k_lo) // 50)  # subsample for speed
    for k in range(k_lo, k_hi + 1, step):
        re = sum(windowed[n] * math.cos(2 * math.pi * k * n / N) for n in range(0, N, 8)) * 8
        im = sum(windowed[n] * math.sin(2 * math.pi * k * n / N) for n in range(0, N, 8)) * 8
        energy += re*re + im*im
    return math.sqrt(energy / max(1, (k_hi - k_lo) // step + 1)) / N

BANDS = {
    'sub_bass':   (20, 80),
    'bass':       (80, 250),
    'low_mid':    (250, 500),
    'mid':        (500, 2000),
    'high_mid':   (2000, 6000),
    'presence':   (6000, 12000),
    'brilliance': (12000, 20000),
}

def spectral_profile(wav, start_sec, dur_sec):
    """Compute normalized spectral profile."""
    bands = {}
    for name, (lo, hi) in BANDS.items():
        bands[name] = band_energy(wav['mono'], wav['rate'], start_sec, dur_sec, lo, hi)
    total = sum(bands.values())
    if total > 0:
        bands = {k: v/total*100 for k, v in bands.items()}
    return bands

# Find first hit in each WAV
def first_hit(mono, threshold=0.05):
    for i, s in enumerate(mono):
        if abs(s) > threshold:
            return i
    return 0

# Drums spectral comparison (300ms from first hit)
d_start = first_hit(drums_desktop['mono']) / drums_desktop['rate']
w_start = first_hit(drums_web['mono']) / drums_web['rate']

d_spec = spectral_profile(drums_desktop, d_start, 0.3)
w_spec = spectral_profile(drums_web, w_start, 0.3)

print('DRUMS — Normalized Spectral Profile (% of total energy):')
print(f'{"Band":>12}  {"Desktop%":>9}  {"Web%":>7}  {"Ratio":>7}')
for band in BANDS:
    d_pct = d_spec.get(band, 0)
    w_pct = w_spec.get(band, 0)
    ratio = w_pct / d_pct if d_pct > 0 else 0
    print(f'{band:>12}  {d_pct:8.1f}%  {w_pct:6.1f}%  {ratio:6.2f}x')

print()

# Clap spectral comparison
d_start = first_hit(clap_desktop['mono']) / clap_desktop['rate']
w_start = first_hit(clap_web['mono']) / clap_web['rate']

d_spec = spectral_profile(clap_desktop, d_start, 0.3)
w_spec = spectral_profile(clap_web, w_start, 0.3)

print('CLAP+FX — Normalized Spectral Profile (% of total energy):')
print(f'{"Band":>12}  {"Desktop%":>9}  {"Web%":>7}  {"Ratio":>7}')
for band in BANDS:
    d_pct = d_spec.get(band, 0)
    w_pct = w_spec.get(band, 0)
    ratio = w_pct / d_pct if d_pct > 0 else 0
    print(f'{band:>12}  {d_pct:8.1f}%  {w_pct:6.1f}%  {ratio:6.2f}x')

## Step 10: Honest Conclusions

What we KNOW, what we INFER, and what we DON'T KNOW.

In [ ]:
print('=' * 70)
print('CONCLUSIONS')
print('=' * 70)
print()
print('WHAT WE KNOW (proven):')
print('  1. Web output is 1.8-2.2x louder than desktop for identical code')
print('  2. The mixer is alive and processing (amp=1 test)')
print('  3. All external gain stages are verified at unity (worklet, Web Audio)')
print('  4. The factor is signal-dependent, NOT a constant scalar')
print('  5. Routing approach and channel count make no difference')
print('  6. Spectral shape is similar (same instrument) but not identical')
print()
print('WHAT WE INFER (process of elimination):')
print('  7. The difference originates inside the scsynth WASM binary')
print('     (all external stages verified → must be internal)')
print()
print('WHAT WE DO NOT KNOW:')
print('  8. The EXACT mechanism inside scsynth WASM')
print('     - Is it PlayBuf output level? (would be signal-independent → unlikely)')
print('     - Is it bus accumulation? (would be constant → unlikely)')
print('     - Is it float32 precision? (could be signal-dependent → plausible)')
print('     - Is it the audio driver path? (desktop has driver, WASM does not)')
print('     - Is it the compiled synthdefs? (not binary-compared)')
print()
print('WHAT SHOULD CHANGE IN SUPERSONIC:')
print('  The honest answer: we do not know the EXACT internal cause.')
print('  We know the SYMPTOM (output is 1.8-2.2x louder) and WHERE it')
print('  originates (inside scsynth WASM, after all external stages are')
print('  eliminated).')
print()
print('  The fix could be:')
print('  a) A worklet-level output gain (pragmatic, treats symptom)')
print('  b) A bug fix in scsynth WASM (correct, treats cause — unknown)')
print('  c) Matching desktop driver behavior in WASM (if driver is the cause)')
print()
print('  We CANNOT recommend a specific code change in SuperSonic without')
print('  knowing the exact internal mechanism. Filing an issue with the')
print('  A/B data is the right next step — Sam Aaron can investigate the')
print('  scsynth WASM internals.')